# Build an OpenQASM program inspector

The QDK's OpenQASM front end exposes two read-only views of a program. Parsing preserves the syntax as written, while semantic analysis resolves names and types, evaluates constants, and expands broadcast gate calls.

In this notebook, one `SOURCE` value moves through both views to answer five practical questions:

1. Did the program parse, and where are its syntax elements and diagnostics?
2. Which declarations, constants, and qubit registers belong to the program?
3. How does the analyzed program differ from what was written?
4. How can those facts become one reusable inspector?
5. What does the inspector report for valid and invalid programs?

These APIs are in preview and may change between QDK releases.

In [ ]:
from typing import Any
from qdk.openqasm import QASMVisitor, parser, semantic
SOURCE = """OPENQASM 3.0;
include "stdgates.inc";
const angle theta = pi / 4;
qubit[2] q;
bit[2] c;
h q;
ctrl @ x q[0], q[1];
rz(theta) q[1];
c = measure q;
"""

result = parser.parse(SOURCE)
print("version:", result.program.version)
print("parse errors:", result.has_errors)
print("diagnostics:", len(result.diagnostics))

## 1. What did the parser read?

The syntax tree preserves each gate call exactly as it appears in `SOURCE`. A visitor avoids assumptions about statement order, while the parse result's source map converts each node's byte span into a one-based line and column for readers.

In [ ]:
class SyntaxGateCollector(QASMVisitor):
    def __init__(self):
        self.gates = []

    def visit_QuantumGate(self, node, source_map):
        source_range = source_map.range_from_span(node.span)
        self.gates.append(
            {
                "name": node.name.name,
                "line": source_range.start.line + 1,
                "column": source_range.start.column + 1,
            }
        )
        self.generic_visit(node, source_map)


syntax_gates = SyntaxGateCollector()
syntax_gates.visit(result.program, result.document.source_map)
syntax_gates.gates

### See parser recovery in action

Parser calls return diagnostics instead of raising, so tools can inspect the recovered tree and show the problem in context. This variant is derived from `SOURCE` by removing one closing bracket. `Diagnostic.render` produces a source-annotated explanation with the offending token underlined.

In [ ]:
def diagnostics_to_plain(result: Any) -> list[dict[str, Any]]:
    diagnostics = []
    source_map = result.document.source_map

    for diagnostic in result.diagnostics:
        location = {"source": None, "line": None, "column": None}
        if diagnostic.labels:
            source_range = source_map.range_from_span(diagnostic.labels[0].span)
            location = {
                "source": source_map.get(source_range.source_id).path,
                "line": source_range.start.line + 1,
                "column": source_range.start.column + 1,
            }

        diagnostics.append({"message": diagnostic.message, **location})

    return diagnostics


SYNTAX_ERROR_SOURCE = SOURCE.replace("q[1];", "q[1;", 1)
syntax_error_result = parser.parse(SYNTAX_ERROR_SOURCE)

assert syntax_error_result.has_errors
print(syntax_error_result.diagnostics[0].render(color=True))

## 2. What does semantic analysis add?

Semantic analysis resolves the declarations that the program can name. The symbol table also contains declarations from `stdgates.inc`, so the inspector uses each symbol's span and the analysis source map to retain only declarations owned by the entry source.

A second variant replaces `theta` with an undefined name. It is valid syntax, but semantic analysis pinpoints the unresolved symbol. This makes the parser-versus-analyzer boundary concrete.

In [ ]:
analysis = semantic.analyze(SOURCE)
analysis_source_map = analysis.document.source_map
entry_source_id = analysis_source_map.entry.id

entry_symbols = [
    symbol
    for symbol in analysis.symbols
    if symbol.span.lo != symbol.span.hi
    and analysis_source_map.range_from_span(symbol.span).source_id == entry_source_id
]

declarations = [
    {
        "name": symbol.name,
        "type": type(symbol.ty).__name__,
        "type_name": symbol.ty.name,
    }
    for symbol in entry_symbols
]

SEMANTIC_ERROR_SOURCE = SOURCE.replace("rz(theta)", "rz(missing_angle)")
semantic_error_parse = parser.parse(SEMANTIC_ERROR_SOURCE)
semantic_error_analysis = semantic.analyze(SEMANTIC_ERROR_SOURCE)

assert not semantic_error_parse.has_errors
assert semantic_error_analysis.has_errors
print(semantic_error_analysis.diagnostics[0].render(color=True))

declarations

### Constants and qubit registers are structured values

Resolved type classes expose properties such as a qubit array's `size`. Constants are Python values; angles additionally expose `radians`. Reading those properties directly keeps the report independent of formatted type or tree dumps.

In [ ]:
def constant_to_plain(value):
    if isinstance(value, semantic.Angle):
        return value.radians
    if isinstance(value, semantic.Duration):
        return {"value": value.value, "unit": str(value.unit)}
    if isinstance(value, complex):
        return {"real": value.real, "imag": value.imag}
    if isinstance(value, (bool, int, float, str)):
        return value
    return None


constants = {
    symbol.name: constant_to_plain(symbol.const_value)
    for symbol in entry_symbols
    if symbol.const_value is not None
}

qubit_registers = [
    {
        "name": symbol.name,
        "size": symbol.ty.size
        if isinstance(symbol.ty, semantic.QubitArrayType)
        else 1,
    }
    for symbol in entry_symbols
    if isinstance(symbol.ty, (semantic.QubitType, semantic.QubitArrayType))
]

print("constants:", constants)
print("qubit registers:", qubit_registers)

## 3. What changed between syntax and semantics?

`QASMVisitor` walks either tree layer. The syntax tree contains the single `h q` call that was written. Semantic analysis resolves `q` as a two-qubit register and expands that broadcast into one `h` operation per qubit. A shared counter makes the difference visible without changing the source.

In [ ]:
class GateCounter(QASMVisitor):
    def __init__(self):
        self.counts = {}

    def visit_QuantumGate(self, node: Any) -> None:
        raw_name = node.name
        name = raw_name if isinstance(raw_name, str) else getattr(raw_name, "name", None)
        if name is not None:
            self.counts[name] = self.counts.get(name, 0) + 1
        self.generic_visit(node)


def gate_counts(program: Any) -> dict[str, int]:
    counter = GateCounter()
    counter.visit(program)
    return counter.counts


syntax_gate_counts = gate_counts(result.program)
semantic_gate_counts = gate_counts(analysis.program)

The two counts answer different questions. Syntax counts describe the source text. Semantic counts describe resolved operations after transformations such as broadcast expansion. Neither view is more correct; the inspector keeps both so callers can choose the level they need.

In [ ]:
print("syntax gates:  ", syntax_gate_counts)
print("semantic gates:", semantic_gate_counts)

assert syntax_gate_counts["h"] == 1
assert semantic_gate_counts["h"] == 2

## 4. Assemble the inspector

The final callable repeats the same stages behind one small interface. Its dictionary shape stays fixed even when parsing or analysis reports errors. Recovered or unavailable facts are skipped, while diagnostics keep `None` for positions that have no source label.

In [ ]:
def entry_symbols_for(analysis: Any) -> list[Any]:
    source_map = analysis.document.source_map
    entry_source_id = source_map.entry.id
    symbols = []

    for symbol in analysis.symbols:
        if symbol.span.lo == symbol.span.hi:
            continue
        try:
            source_id = source_map.range_from_span(symbol.span).source_id
        except ValueError:
            continue
        if source_id == entry_source_id:
            symbols.append(symbol)

    return symbols

The report keeps parser and analyzer error states separate because syntax can recover successfully while semantic checks still find unresolved names or type errors. Analyzer diagnostics can repeat parser diagnostics, so the callable deduplicates equal plain-data entries before returning them.

In [ ]:
def inspect_openqasm(source: str) -> dict[str, Any]:
    parsed = parser.parse(source)
    analyzed = semantic.analyze(source)
    symbols = entry_symbols_for(analyzed)

    declarations = [
        {
            "name": symbol.name,
            "type": type(symbol.ty).__name__,
            "type_name": symbol.ty.name,
        }
        for symbol in symbols
    ]

    constants = {}
    for symbol in symbols:
        if symbol.const_value is None:
            continue
        value = constant_to_plain(symbol.const_value)
        if value is not None:
            constants[symbol.name] = value

    qubit_registers = [
        {
            "name": symbol.name,
            "size": symbol.ty.size
            if isinstance(symbol.ty, semantic.QubitArrayType)
            else 1,
        }
        for symbol in symbols
        if isinstance(symbol.ty, (semantic.QubitType, semantic.QubitArrayType))
    ]

    diagnostics = diagnostics_to_plain(parsed)
    for diagnostic in diagnostics_to_plain(analyzed):
        if diagnostic not in diagnostics:
            diagnostics.append(diagnostic)

    return {
        "has_parse_errors": parsed.has_errors,
        "has_analysis_errors": analyzed.has_errors,
        "diagnostics": diagnostics,
        "declarations": declarations,
        "constants": constants,
        "qubit_registers": qubit_registers,
        "syntax_gate_counts": gate_counts(parsed.program),
        "semantic_gate_counts": gate_counts(analyzed.program),
    }

## 5. Compare all three outcomes

The final inspector receives the valid program and both variants derived from it. The compact error matrix shows why separate parser and analyzer status fields matter, while the earlier rendered diagnostics provide the detailed explanation a developer would see.

In [ ]:
from pprint import pprint

reports = {
    "valid": inspect_openqasm(SOURCE),
    "syntax_error": inspect_openqasm(SYNTAX_ERROR_SOURCE),
    "semantic_error": inspect_openqasm(SEMANTIC_ERROR_SOURCE),
}

assert not reports["valid"]["has_parse_errors"]
assert not reports["valid"]["has_analysis_errors"]
assert {item["name"] for item in reports["valid"]["declarations"]} == {"theta", "q", "c"}
assert reports["valid"]["syntax_gate_counts"]["h"] == 1
assert reports["valid"]["semantic_gate_counts"]["h"] == 2
assert reports["syntax_error"]["has_parse_errors"]
assert not reports["semantic_error"]["has_parse_errors"]
assert reports["semantic_error"]["has_analysis_errors"]

error_matrix = {
    name: {
        "parse_errors": report["has_parse_errors"],
        "analysis_errors": report["has_analysis_errors"],
        "diagnostics": len(report["diagnostics"]),
    }
    for name, report in reports.items()
}

pprint(
    {
        "error_matrix": error_matrix,
        "valid_program": reports["valid"],
    },
    indent=4,
    sort_dicts=False,
    width=1,
)

## Where to go next

The inspector is intentionally read-only. Use `help(qdk.openqasm.parser)` and `help(qdk.openqasm.semantic)` to explore custom include resolution, canonical syntax serialization, and additional node types. The [OpenQASM interop notebook](./openqasm.ipynb) covers running, compiling, and estimating programs.